# FLUX.2 klein 4B — Full Precision vs 8-bit vs 4-bit on T4
Tests the same prompt/seed at three precision levels, with memory cleanup between each. Reports load time, generation time, peak VRAM, and saves each image. Set Runtime to T4 GPU first.

In [ ]:
!pip install -q -U diffusers transformers accelerate sentencepiece protobuf bitsandbytes

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import torch, gc, time
from diffusers import Flux2KleinPipeline, PipelineQuantizationConfig

MODEL_ID = "black-forest-labs/FLUX.2-klein-4B"
SEED = 42
PROMPT = ("Chiku and Pinku starting their adventure in the garden. Style: Classic 1990s hand-drawn animated jungle film with clean ink outlines and cel shading. Chiku, the small sleek cat with soft gray fur. Pointed ears. Sharp green eyes. Long graceful tail. Four agile legs. Standing on grass. Pinku, the friendly dog with golden brown fur. Floppy ears. Bright loyal eyes. Wagging tail. Four legs. Running beside Chiku. Both characters visible. Jungle garden setting with lush green grass and bushes. Afternoon sunlight filtering through leaves. Light beige ground. The scene shows excitement and adventure.")

def cleanup():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

def load_pipe(precision):
    if precision == "full":
        pipe = Flux2KleinPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)
    elif precision == "8bit":
        qc = PipelineQuantizationConfig(quant_backend="bitsandbytes_8bit", quant_kwargs={"load_in_8bit": True}, components_to_quantize=["transformer"])
        pipe = Flux2KleinPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, quantization_config=qc)
    elif precision == "4bit":
        qc = PipelineQuantizationConfig(quant_backend="bitsandbytes_4bit", quant_kwargs={"load_in_4bit": True, "bnb_4bit_quant_type": "nf4", "bnb_4bit_compute_dtype": torch.bfloat16}, components_to_quantize=["transformer"])
        pipe = Flux2KleinPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, quantization_config=qc)
    else:
        raise ValueError(precision)
    pipe.enable_model_cpu_offload()
    return pipe

results = {}
for precision in ["full", "8bit", "4bit"]:
    print(f"\n=== Testing {precision} precision ===")
    cleanup()
    load_start = time.time()
    try:
        pipe = load_pipe(precision)
    except Exception as e:
        print(f"FAILED to load {precision}: {e}"); results[precision] = {"error": str(e)}; continue
    load_time = time.time() - load_start
    torch.cuda.reset_peak_memory_stats()
    gen_start = time.time()
    try:
        image = pipe(prompt=PROMPT, height=1024, width=1024, num_inference_steps=4, guidance_scale=1.0, generator=torch.Generator(device="cuda").manual_seed(SEED)).images[0]
    except Exception as e:
        print(f"FAILED to generate at {precision}: {e}"); results[precision] = {"error": str(e)}; del pipe; cleanup(); continue
    gen_time = time.time() - gen_start
    peak_vram_gb = torch.cuda.max_memory_allocated() / 1e9
    fname = f"klein_{precision}.png"
    image.save(fname)
    results[precision] = {"load_time_s": round(load_time,1), "gen_time_s": round(gen_time,2), "peak_vram_gb": round(peak_vram_gb,2), "image_path": fname}
    print(f"Load: {load_time:.1f}s | Generation: {gen_time:.2f}s | Peak VRAM: {peak_vram_gb:.2f} GB | Saved: {fname}")
    del pipe; cleanup()

In [ ]:
import pandas as pd
df = pd.DataFrame(results).T
df